# HW01 · Регрессия и компромисс смещение–дисперсия

Три части: (a) концептуальные упражнения ISLP 3.7, (b) две подгонки одной и той
же модели ROA — как вывод и как предсказание, (c) 250 слов о том, почему два
критерия расходятся. Заглушки помечены `TODO(hw1-…)`.

Ваш **протокол валидации** здесь ещё простой (разбиение по времени из L02). В S3
вы напишете его как отдельный документ; пока держитесь принципа «делить до того,
как подгонять».

**Единицы:** тысячи USD; `e_roa` аннуализирован. Код и ответы — по-английски.

In [ ]:
# HW01 · Regression and the bias-variance trade-off
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Данные лежат в репозитории курса. В Colab читаем их прямо по ссылке;
# если ноутбук запущен из склонированного репозитория — с диска.
DATA = "https://raw.githubusercontent.com/nvvoitov/da_intro/main/data/"
if os.path.exists("../data/banks_panel.parquet"):
    DATA = "../data/"

SEED = 20260201
rng = np.random.default_rng(SEED)      # один генератор случайности на весь ноутбук

# Косметика: единый вид таблиц и графиков. Вникать не обязательно.
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.4g}")
plt.rcParams.update({"figure.figsize": (7, 4.2), "figure.dpi": 110, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False, "font.size": 11,
                     "axes.spines.right": False, "legend.frameon": False,
                     "axes.titleweight": "bold"})
BLUE, ORANGE, GREY = "#0072B2", "#D55E00", "#444444"

panel = pd.read_parquet(DATA + "banks_panel.parquet")
print(panel.shape, "| банков:", panel.cert.nunique())

## 2 · Концепты — Conceptual (ISLP §3.7, exercises 3 and 4)

`TODO(hw1-a)`. Ответьте на упражнения **3** и **4** из ISLP §3.7. Это про
интерпретацию взаимодействий и про сравнение линейной и кубической подгонок при
неизвестной истинной функции — ровно то, что делает §4 лекции наглядным.
Формулы — в LaTeX; словами объясните, а не только посчитайте.

> _Your answer here._

## 3 · Две подгонки — The two fits

`TODO(hw1-b)`. На одном и том же наборе признаков CAMELS (капитал, качество
активов, менеджмент, ликвидность — **без** блока `e_`) подгоните ROA дважды:

1. **вывод** — `statsmodels` OLS с SE, кластеризованными по `cert`; сохраните
   результат в `ols`;
2. **предсказание** — на разбиении по времени (как в L02 §5:
   `train = (df.repdte <= "2015-12-31").to_numpy()`, `test = ~train`);
   посчитайте held-out `rmse` (float, в единицах ROA).

Return: `ols` (fitted results) и `rmse` (float). Отклонитесь от протокола L02 —
объясните, почему, в §4.

In [ ]:
# TODO(hw1-b): fit ROA twice on the same CAMELS features.
# Return: `ols`  — a fitted statsmodels result, SEs clustered by cert
#         `rmse` — a positive float, held-out RMSE on a temporal split
# Do not use the e_ (earnings) block as a feature — it *is* ROA.
raise NotImplementedError

## 4 · Почему критерии расходятся — Why the two criteria disagree

`TODO(hw1-c)`. ~250 слов (по-английски): какой критерий вы бы использовали для
вопроса «*почему* ROA такой?», а какой — для «*каким будет* ROA нового банка?»,
и почему «лучшая» спецификация может отличаться. Свяжите с §4 лекции (смещение
против дисперсии) и с тем, что кластеризация SE сделала со значимостью в §3.

> _Your answer here (250 words)._

## Self-check

Механическая проверка контракта (не оценка смысла). Запустите после §3.

In [ ]:
assert hasattr(ols, "params") and hasattr(ols, "bse"), \
    "hw1-b: ожидается результат smf.ols(...).fit(...)"
assert getattr(ols, "cov_type", "nonrobust") == "cluster", \
    "hw1-b: SE должны быть кластеризованы: fit(cov_type='cluster', cov_kwds={'groups': ...})"
assert int(ols.nobs) > 1000, "hw1-b: подозрительно мало наблюдений — проверьте dropna"
print(f"OK · вывод: {int(ols.nobs):,} наблюдений, cov_type={ols.cov_type}")

rmse = float(rmse)
assert np.isfinite(rmse) and rmse > 0, "hw1-b: rmse должен быть положительным числом"
assert rmse < 0.05, f"hw1-b: RMSE={rmse:.4f} велик для ROA в долях — та ли цель?"
print(f"OK · прогноз: held-out RMSE={rmse:.5f}")